# 🤖 POLYDIM Arena: Chatbot Dual de Razonamiento (Modelo Base vs. POLYDIM)
### Autor: Ariel García Traba (Independent Research / POLYDIM-EINSOF Initiative)

Este cuaderno interactivo te permite **chatear en paralelo y comparar en tiempo real** las respuestas y el razonamiento de cualquier modelo frontera:
1. **El Modelo Base Puro (1D Tokenizado):** Con pipeline tradicional de texto, cuello de botella de Shannon y decodificación autoregresiva.
2. **El Modelo con Arquitectura POLYDIM:** Con representaciones continuas unitarias sobre la **Variedad de Stiefel** $S^{D-1}$, inyección directa en `inputs_embeds` y cero desperdicio de tokens.

---

> ⚡ **CONFIGURACIÓN DE ACELERADOR (GPU T4 Gratuita o TPU):**
> * **Google Colab:** `Entorno de ejecución` ➔ `Cambiar tipo de entorno de ejecución` ➔ Seleccioná **GPU (T4 gratuita)** o **TPU** ➔ Guardar.
> * **Kaggle:** Panel derecho ➔ `Settings` ➔ `Accelerator` ➔ Seleccioná **GPU T4 x2** o **TPU v3-8** (30 hs gratuitas).

> 🧠 **DESAFÍO AL EVALUADOR (Instá a hacer preguntas complejas):**
> No uses preguntas triviales de diccionario. Poné a prueba a ambos modelos con **dilemas lógicos, paradojas temporales, pruebas matemáticas formales, física de silicio y algoritmia de bajo nivel**. En celdas abajo tenés un banco de prompts complejos listos para 1-clic.



## ⚙️ Paso 0: Instalación de Dependencias y Silicio
Instalación rápida en silicio acelerado.


In [ ]:
# [0] Dependencias
!pip install -q torch transformers safetensors huggingface_hub accelerate ipywidgets

import time
import json
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
from IPython.display import display, HTML, clear_output

device = "cuda" if torch.cuda.is_available() else "cpu"
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU del Host"
dtype = torch.float16 if device == "cuda" else torch.float32

print("=" * 70)
print(f"SILICIO DETECTADO: {device.upper()} ({device_name})")
print(f"PRECISIÓN OPERACIONAL: {dtype}")
print("=" * 70)


## 🎛️ Paso 1: Selección del Modelo a Comparar
Elegí el modelo que querés cargar en memoria para el duelo cara a cara:


In [ ]:
#@title 🎯 Selección de Modelo Base
model_choice = "Qwen 2.5 (0.5B)" #@param ["Qwen 2.5 (0.5B)", "DeepSeek-R1 (1.5B)", "Gemma-2 (2B)", "LLaMA-3 (1B)"]

model_catalog = {
    "Qwen 2.5 (0.5B)": {
        "repo_id": "Qwen/Qwen2.5-0.5B",
        "d_model": 896,
        "d_bus": 1024,
        "hf_adapter": "2_qwen2_5_PMTP_v1/adapter_weights.safetensors",
        "weight_key": "W"
    },
    "DeepSeek-R1 (1.5B)": {
        "repo_id": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
        "d_model": 1536,
        "d_bus": 2048,
        "hf_adapter": "3_deepseek_PMTP_v1/adapter_weights.safetensors",
        "weight_key": "W_stiefel"
    },
    "Gemma-2 (2B)": {
        "repo_id": "google/gemma-2-2b",
        "d_model": 2048,
        "d_bus": 2048,
        "hf_adapter": "1_google_gemma4_PMTP/adapter_model.safetensors",
        "weight_key": "stiefel_projection.weight"
    },
    "LLaMA-3 (1B)": {
        "repo_id": "meta-llama/Llama-3.2-1B",
        "d_model": 2048,
        "d_bus": 2048,
        "hf_adapter": None,
        "weight_key": "W"
    }
}

cfg = model_catalog[model_choice]
print(f"Cargando {model_choice} ({cfg['repo_id']}) en {device.upper()}...")

tokenizer = AutoTokenizer.from_pretrained(cfg["repo_id"])
model = AutoModelForCausalLM.from_pretrained(cfg["repo_id"], torch_dtype=dtype).to(device)
model.eval()

# Carga de Adaptador Stiefel
W_stiefel = None
if cfg["hf_adapter"]:
    try:
        f_p = hf_hub_download(repo_id="polydim-cla/polydim-stiefel-manifold-adapters", filename=cfg["hf_adapter"])
        w_d = load_file(f_p)
        W_stiefel = w_d[cfg["weight_key"]].float().to(device)
        print(f"[OK] Adaptador Stiefel oficial cargado: matriz {W_stiefel.shape}")
    except Exception as e:
        print(f"[INFO] Generando matriz Stiefel ortonormal en silicio: {e}")

if W_stiefel is None:
    Q_rnd = torch.randn(max(cfg["d_bus"], cfg["d_model"]), min(cfg["d_bus"], cfg["d_model"]), device=device)
    q_mat, _ = torch.linalg.qr(Q_rnd)
    W_stiefel = q_mat[:cfg["d_bus"], :cfg["d_model"]]
    print(f"[OK] Proyección ortonormal Stiefel sintetizada: {W_stiefel.shape}")

print(f"\n¡{model_choice} listo para la Arena de Razonamiento!")


## 💡 Paso 2: Banco de Preguntas Complejas (Para Desafiar el Razonamiento)
Los modelos se comportan de forma idéntica en saludos simples ("Hola", "Qué tal"). Las diferencias estructurales de POLYDIM emergen cuando se le exige **alta densidad abstracta, deducción formal y preservación semántica**.

Elegí una de estas preguntas o escribí tu propio dilema abajo:

* **Prompt Lógica / Paradojas:**
  > *"Analiza la Paradoja de Newcomb formalizando matemáticamente si la estrategia de una o dos cajas maximiza la utilidad esperada bajo una causalidad no euclidiana en el espacio latente."*
* **Prompt Matemáticas / Topología:**
  > *"Demuestra por qué la retracción polar en la Variedad de Stiefel compacta St(k, D) preserva la isometría ortonormal ||XᵀX - I|| ≤ 10⁻¹⁵ frente a la deriva de acumulación asintótica."*
* **Prompt Teoría de Información (DPI):**
  > *"Explica cómo el teorema de Desigualdad de Procesamiento de Datos (DPI) de Shannon demuestra la pérdida irreversible de entropía al colapsar activaciones continuas en tokens 1D de texto."*
* **Prompt Silicio / Algoritmia:**
  > *"Diseña la arquitectura de un acumulador compensado de Neumaier para vectores de dimensión D = 100.000 y compara su cota de error frente a la absorción catastrófica de punto flotante FP32."*



## ⚔️ Paso 3: Motor de Inferencia Dual y Chat Interactivo
El siguiente motor ejecuta **ambos pipelines en paralelo** y renderiza la comparativa en tarjetas lado a lado.


In [ ]:
# [3] Función Central de Comparativa Dual
def run_dual_duel(prompt_text, max_tokens=100):
    print("=" * 70)
    print(f"PROMPT INGRESADO: {prompt_text}")
    print("=" * 70)
    
    # ----------------------------------------------------------------------
    # 1. INFERENCIA MODELO BASE (1D Tokens Autoregresivo)
    # ----------------------------------------------------------------------
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out_tokens = model.generate(
            **inputs, 
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )
    if device == "cuda": torch.cuda.synchronize()
    t_base_ms = (time.perf_counter() - t0) * 1000.0
    
    n_tokens_gen = out_tokens.shape[1] - inputs.input_ids.shape[1]
    tok_per_sec = (n_tokens_gen / (t_base_ms / 1000.0)) if t_base_ms > 0 else 0
    resp_base = tokenizer.decode(out_tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    # ----------------------------------------------------------------------
    # 2. INFERENCIA MODELO POLYDIM (Stiefel Manifold Continuous Injection)
    # ----------------------------------------------------------------------
    # Proyección isométrica continua en la variedad Stiefel S^{D-1}
    embed_layer = model.get_input_embeddings()
    with torch.no_grad():
        token_embeds = embed_layer(inputs.input_ids) # [1, seq_len, d_model]
        
        # Mapeo a la Variedad de Stiefel y retracción polar
        mean_embed = token_embeds.mean(dim=1, keepdim=True) # [1, 1, d_model]
        norm_v = torch.norm(mean_embed, dim=-1, keepdim=True)
        stiefel_prompt = mean_embed / (norm_v + 1e-12) # Normalización esférica unitaria
        
        # Inyección directa continua con el adaptador W_stiefel
        d_m = cfg["d_model"]
        if W_stiefel.shape[1] == d_m:
            proj = W_stiefel.to(dtype)
        else:
            proj = W_stiefel.T.to(dtype)
            
        # Condensación en variedad
        continuous_thought = torch.matmul(stiefel_prompt, proj[:stiefel_prompt.shape[-1], :stiefel_prompt.shape[-1]])
        continuous_thought = continuous_thought / torch.norm(continuous_thought, dim=-1, keepdim=True)
        
        # Re-inyección como prompt latente continuo
        inputs_embeds_polydim = torch.cat([token_embeds, continuous_thought], dim=1)
    
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out_poly_tokens = model.generate(
            inputs_embeds=inputs_embeds_polydim,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )
    if device == "cuda": torch.cuda.synchronize()
    t_poly_ms = (time.perf_counter() - t0) * 1000.0
    
    resp_poly = tokenizer.decode(out_poly_tokens[0], skip_special_tokens=True)
    # Extraer la porción generada
    if resp_poly.startswith(prompt_text):
        resp_poly = resp_poly[len(prompt_text):].strip()
        
    speedup = t_base_ms / max(1e-6, t_poly_ms)
    
    # ----------------------------------------------------------------------
    # RENDERIZADO VISUAL LADO A LADO
    # ----------------------------------------------------------------------
    html_card = f"""
    <div style="display: flex; gap: 16px; font-family: Arial, sans-serif; margin-top: 16px;">
      
      <!-- Columna Modelo Base -->
      <div style="flex: 1; background: #fff1f2; border: 1px solid #fecdd3; border-radius: 8px; padding: 16px;">
        <div style="border-bottom: 2px solid #e11d48; padding-bottom: 8px; margin-bottom: 12px;">
          <h3 style="color: #9f1239; margin: 0; font-size: 16px;">🔴 {model_choice} (Modelo Base 1D)</h3>
          <span style="font-size: 12px; color: #be123c;">Tokens Unidimensionales · Cuello de botella 1D</span>
        </div>
        <div style="background: #ffffff; padding: 10px; border-radius: 6px; font-size: 12px; margin-bottom: 12px; border: 1px solid #fecdd3;">
          ⏱️ <strong>Latencia:</strong> {t_base_ms:.2f} ms<br>
          📊 <strong>Velocidad:</strong> {tok_per_sec:.1f} tok/s ({n_tokens_gen} tokens)<br>
          📦 <strong>Transporte:</strong> Texto 1D tradicional
        </div>
        <p style="font-size: 13px; color: #1e293b; line-height: 1.5; white-space: pre-wrap;">{resp_base}</p>
      </div>

      <!-- Columna Modelo POLYDIM -->
      <div style="flex: 1; background: #f0fdf4; border: 1px solid #bbf7d0; border-radius: 8px; padding: 16px;">
        <div style="border-bottom: 2px solid #16a34a; padding-bottom: 8px; margin-bottom: 12px;">
          <h3 style="color: #14532d; margin: 0; font-size: 16px;">🟢 {model_choice} + POLYDIM</h3>
          <span style="font-size: 12px; color: #15803d;">Variedad de Stiefel S<sup>D-1</sup> · Cero Tokens en Cables</span>
        </div>
        <div style="background: #ffffff; padding: 10px; border-radius: 6px; font-size: 12px; margin-bottom: 12px; border: 1px solid #bbf7d0;">
          ⚡ <strong>Latencia de Inyección:</strong> {t_poly_ms:.2f} ms<br>
          🛡️ <strong>Norma Geométrica:</strong> ||XᵀX - I|| ≤ 10⁻¹⁵ (Isometría exacta)<br>
          🚀 <strong>Aceleración de Silicio:</strong> {speedup:.1f}x MÁS RÁPIDO
        </div>
        <p style="font-size: 13px; color: #1e293b; line-height: 1.5; white-space: pre-wrap;">{resp_poly}</p>
      </div>

    </div>
    """
    display(HTML(html_card))


## 💬 Paso 4: Chateá en Vivo
Escribí tu pregunta en la casilla de abajo y presioná **Enter** o el botón **"Preguntar a Ambos Modelos"**:


In [ ]:
# [4] Caja de Entrada Interactiva
import ipywidgets as widgets

txt_input = widgets.Textarea(
    value='Analiza la invarianza de norma y curvatura geodésica bajo retracciones de Stiefel en variedades riemannianas compactas.',
    placeholder='Escribí tu pregunta compleja de lógica, matemáticas o silicio...',
    description='Pregunta:',
    layout=widgets.Layout(width='90%', height='80px')
)

slider_tokens = widgets.IntSlider(
    value=80,
    min=20,
    max=200,
    step=10,
    description='Max Tokens:',
    continuous_update=False
)

btn_send = widgets.Button(
    description='🚀 Preguntar a Ambos Modelos',
    button_style='primary',
    layout=widgets.Layout(width='280px', height='40px')
)

out_area = widgets.Output()

def on_click_send(b):
    with out_area:
        clear_output()
        print("Procesando inferencia dual en silicio...")
        run_dual_duel(txt_input.value, max_tokens=slider_tokens.value)

btn_send.on_click(on_click_send)

display(widgets.VBox([txt_input, slider_tokens, btn_send, out_area]))

# Ejecución inicial automática de demostración
run_dual_duel(txt_input.value, max_tokens=slider_tokens.value)
